# STAGE 1 DEV RESULT

In [1]:
import os

print("=== Checking all input files ===")
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

=== Checking all input files ===
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/train.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/requirements.txt
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/predict.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/resume_training.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/utils/prepare_clc_fce_data.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/utils/helpers.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/utils/preprocess_data.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/gector/seq2labels_model.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/gector/trainer.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/gector/datareader.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/gector/gec_model.py
/kaggle/input/datasets/roseannnnnaguil

In [2]:
import os
import shutil

code_path = "/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code"
work_dir  = "/kaggle/working/gector"

if os.path.exists(work_dir):
    shutil.rmtree(work_dir)
shutil.copytree(code_path, work_dir)
print("✅ Code copied!")

for root, dirs, files in os.walk(work_dir):
    for file in files:
        print(os.path.join(root, file))

✅ Code copied!
/kaggle/working/gector/requirements.txt
/kaggle/working/gector/resume_training.py
/kaggle/working/gector/predict.py
/kaggle/working/gector/train.py
/kaggle/working/gector/utils/prepare_clc_fce_data.py
/kaggle/working/gector/utils/preprocess_data.py
/kaggle/working/gector/utils/helpers.py
/kaggle/working/gector/gector/gec_model.py
/kaggle/working/gector/gector/datareader.py
/kaggle/working/gector/gector/trainer.py
/kaggle/working/gector/gector/seq2labels_model.py
/kaggle/working/gector/data/verb-form-vocab.txt
/kaggle/working/gector/data/output_vocabulary/labels.txt
/kaggle/working/gector/data/output_vocabulary/d_tags.txt
/kaggle/working/gector/data/output_vocabulary/non_padded_namespaces.txt


In [3]:
new_gec_model = '''
import logging
import os
from time import time

import torch
from transformers import AutoTokenizer

from gector.seq2labels_model import Seq2Labels
from utils.helpers import PAD, UNK, get_target_sent_by_edits

logger = logging.getLogger(__file__)

MODEL_PATHS = {
    "bert":          "bert-base-cased",
    "distilbert":    "distilbert-base-uncased",
    "albert":        "albert-base-v2",
    "roberta":       "roberta-base",
    "gpt2":          "gpt2",
    "transformerxl": "transfo-xl-wt103",
    "xlnet":         "xlnet-base-cased",
    "tinybert":      "huawei-noah/TinyBERT_General_4L_312D",
    "mobilebert":    "google/mobilebert-uncased",
}


class SimpleVocab:
    def __init__(self, vocab_path):
        self.token2idx = {}
        self.idx2token = {}
        labels_file = os.path.join(vocab_path, "labels.txt")
        with open(labels_file, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                token = line.strip()
                self.token2idx[token] = idx
                self.idx2token[idx] = token

    def get_token_index(self, token, namespace=None):
        return self.token2idx.get(token, 0)

    def get_token_from_index(self, idx, namespace=None):
        return self.idx2token.get(idx, UNK)

    def __len__(self):
        return len(self.token2idx)


class GecBERTModel(object):
    def __init__(self, vocab_path=None, model_paths=None,
                 weigths=None,
                 max_len=50,
                 min_len=3,
                 lowercase_tokens=False,
                 log=False,
                 iterations=5,
                 min_probability=0.0,
                 model_name="tinybert",
                 special_tokens_fix=0,
                 is_ensemble=False,
                 min_error_probability=0.0,
                 confidence=0.0,
                 resolve_cycles=False):

        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.max_len = max_len
        self.min_len = min_len
        self.lowercase_tokens = lowercase_tokens
        self.min_probability = min_probability
        self.min_error_probability = min_error_probability
        self.log = log
        self.iterations = iterations
        self.confidence = confidence
        self.model_weights = list(map(float, weigths)) if weigths else [1] * len(model_paths)

        self.vocab = SimpleVocab(vocab_path)
        num_labels = len(self.vocab)

        weights_name = MODEL_PATHS.get(model_name, model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(weights_name)

        self.models = []
        for model_path in model_paths:
            model = Seq2Labels(
                encoder_name=weights_name,
                num_labels_classes=num_labels,
                num_detect_classes=4,       
                confidence=confidence,
                special_tokens_fix=0,        
                tune_bert=False,
            ).to(self.device)
            state = torch.load(model_path, map_location=self.device)
            model.load_state_dict(state)
            model.eval()
            self.models.append(model)

        print(f"✅ Loaded {len(self.models)} model(s) on {self.device}")

    def preprocess(self, token_batch):
        seq_lens = [len(s) for s in token_batch if s]
        if not seq_lens:
            return None

        all_input_ids, all_attention_masks, all_offsets = [], [], []

        for sequence in token_batch:
            tokens = sequence[:self.max_len - 1]
            words  = ["$START"] + tokens

            input_ids = []
            offsets   = []
            for word in words:
                word_ids = self.tokenizer.encode(word, add_special_tokens=False)
                if not word_ids:
                    word_ids = [self.tokenizer.unk_token_id]
                offsets.append(len(input_ids))
                input_ids.extend(word_ids)

            cls_id    = self.tokenizer.cls_token_id
            sep_id    = self.tokenizer.sep_token_id
            input_ids = [cls_id] + input_ids + [sep_id]
            offsets   = [o + 1 for o in offsets]

            all_input_ids.append(input_ids)
            all_offsets.append(offsets)

        max_seq_len  = max(len(ids) for ids in all_input_ids)
        max_word_len = max(len(off) for off in all_offsets)
        pad_id = self.tokenizer.pad_token_id or 0

        padded_ids, attn_masks, padded_offsets = [], [], []
        for ids, offsets in zip(all_input_ids, all_offsets):
            pad_len = max_seq_len - len(ids)
            padded_ids.append(ids + [pad_id] * pad_len)
            attn_masks.append([1] * len(ids) + [0] * pad_len)
            off_pad = max_word_len - len(offsets)
            padded_offsets.append(offsets + [0] * off_pad)

        return {
            "input_ids":      torch.tensor(padded_ids,     dtype=torch.long).to(self.device),
            "attention_mask": torch.tensor(attn_masks,     dtype=torch.long).to(self.device),
            "offsets":        torch.tensor(padded_offsets, dtype=torch.long).to(self.device),
        }

    def predict(self, batch):
        t0 = time()
        predictions = []
        for model in self.models:
            with torch.no_grad():
                output = model.forward(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    offsets=batch["offsets"],
                )
            predictions.append(output)
        probs, idx, error_probs = self._convert(predictions)
        if self.log:
            print(f"Inference time {time() - t0:.2f}s")
        return probs, idx, error_probs

    def _convert(self, data):
        all_class_probs = torch.zeros_like(data[0]["class_probabilities_labels"])
        error_probs     = torch.zeros_like(data[0]["max_error_probability"])
        for output, weight in zip(data, self.model_weights):
            all_class_probs += weight * output["class_probabilities_labels"] / sum(self.model_weights)
            error_probs     += weight * output["max_error_probability"]      / sum(self.model_weights)
        max_vals = torch.max(all_class_probs, dim=-1)
        return max_vals[0].tolist(), max_vals[1].tolist(), error_probs.tolist()

    def get_token_action(self, token, index, prob, sugg_token):
        if prob < self.min_probability or sugg_token in [UNK, PAD, "$KEEP"]:
            return None
        if sugg_token.startswith("$REPLACE_") or sugg_token.startswith("$TRANSFORM_") or sugg_token == "$DELETE":
            start_pos, end_pos = index, index + 1
        elif sugg_token.startswith("$APPEND_") or sugg_token.startswith("$MERGE_"):
            start_pos = end_pos = index + 1
        else:
            return None
        sugg_token_clear = "" if sugg_token == "$DELETE" else (
            sugg_token if sugg_token.startswith("$TRANSFORM_") or sugg_token.startswith("$MERGE_")
            else sugg_token[sugg_token.index("_") + 1:]
        )
        return start_pos - 1, end_pos - 1, sugg_token_clear, prob

    def postprocess_batch(self, batch, all_probabilities, all_idxs, error_probs):
        all_results = []
        noop_index = self.vocab.get_token_index("$KEEP", "labels")
        for tokens, probabilities, idxs, error_prob in zip(
                batch, all_probabilities, all_idxs, error_probs):
            length = min(len(tokens), self.max_len)
            if max(idxs) == 0 or error_prob < self.min_error_probability:
                all_results.append(tokens)
                continue
            edits = []
            for i in range(length):
                token = tokens[i - 1]
                if idxs[i] == noop_index:
                    continue
                sugg_token = self.vocab.get_token_from_index(idxs[i], "labels")
                action = self.get_token_action(token, i, probabilities[i], sugg_token)
                if action:
                    edits.append(action)
            all_results.append(get_target_sent_by_edits(tokens, edits))
        return all_results

    def update_final_batch(self, final_batch, pred_ids, pred_batch, prev_preds_dict):
        new_pred_ids  = []
        total_updated = 0
        for i, orig_id in enumerate(pred_ids):
            orig       = final_batch[orig_id]
            pred       = pred_batch[i]
            prev_preds = prev_preds_dict[orig_id]
            if orig != pred and pred not in prev_preds:
                final_batch[orig_id] = pred
                new_pred_ids.append(orig_id)
                prev_preds_dict[orig_id].append(pred)
                total_updated += 1
            elif orig != pred and pred in prev_preds:
                final_batch[orig_id] = pred
                total_updated += 1
        return final_batch, new_pred_ids, total_updated

    def handle_batch(self, full_batch):
        final_batch     = full_batch[:]
        batch_size      = len(full_batch)
        prev_preds_dict = {i: [final_batch[i]] for i in range(batch_size)}
        short_ids       = [i for i in range(batch_size) if len(full_batch[i]) < self.min_len]
        pred_ids        = [i for i in range(batch_size) if i not in short_ids]
        total_updates   = 0

        for n_iter in range(self.iterations):
            orig_batch = [final_batch[i] for i in pred_ids]
            sequences  = self.preprocess(orig_batch)
            if not sequences:
                break
            probabilities, idxs, error_probs = self.predict(sequences)
            pred_batch = self.postprocess_batch(orig_batch, probabilities, idxs, error_probs)
            if self.log:
                print(f"Iteration {n_iter + 1}: {round(100*len(pred_ids)/batch_size, 1)}% of sentences active")
            final_batch, pred_ids, cnt = self.update_final_batch(
                final_batch, pred_ids, pred_batch, prev_preds_dict)
            total_updates += cnt
            if not pred_ids:
                break

        return final_batch, total_updates
'''

import os
with open(f"{work_dir}/gector/gec_model.py", "w") as f:
    f.write(new_gec_model)
print("✅ gec_model.py!")

✅ gec_model.py!


In [4]:
content = open(f"{work_dir}/predict.py").read()
content = content.replace(
    "choices=['bert', 'gpt2', 'transformerxl', 'xlnet', 'distilbert', 'roberta', 'albert'],",
    ""
)
content = content.replace(
    "default='roberta'",
    "default='tinybert'"
)
with open(f"{work_dir}/predict.py", "w") as f:
    f.write(content)
print("✅ predict.py updated!")

✅ predict.py updated!


In [5]:
import os
os.system("pip install transformers torch tqdm python-Levenshtein -q")
os.system("pip install errant -q")
print("✅ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.3/499.3 kB 9.2 MB/s eta 0:00:00
✅ Dependencies installed!


In [6]:
import os

data_path = "/kaggle/input/datasets/roseannnnnaguilar/gec-gector-datasets-new/gec_gector_datasets_new"
test_path = "/kaggle/input/datasets/roseannnnnaguilar/gec-test-data/gec_test_data"
work_dir  = "/kaggle/working/gector"

VOCAB_PATH   = "/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/data/output_vocabulary"
GOLD_M2      = f"{test_path}/ABCN.dev.gold.bea19.m2"

STAGE1_DIR   = "/kaggle/working/models/stage1"
PRED_OUTPUT  = "/kaggle/working/predictions.txt"
HYP_M2       = "/kaggle/working/hyp.m2"

os.makedirs(STAGE1_DIR, exist_ok=True)

print("✅ Paths defined!")
print("\nVerifying files:")
for label, path in [
    ("Vocab",    VOCAB_PATH),
    ("Gold M2",  GOLD_M2),
]:
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"  {status} {label}: {path}")


✅ Paths defined!

Verifying files:
  ✅ Vocab: /kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/data/output_vocabulary
  ✅ Gold M2: /kaggle/input/datasets/roseannnnnaguilar/gec-test-data/gec_test_data/ABCN.dev.gold.bea19.m2


In [7]:
import shutil
import os


stage1_checkpoint = "/kaggle/input/datasets/roseannnnnaguilar/stage1-final-cp/stage1_final_cp"

print("Restoring Stage 1 best model...")
src = os.path.join(stage1_checkpoint, "model_best.th") 
dst = os.path.join(STAGE1_DIR, "model_best.th")
shutil.copy(src, dst)
size = os.path.getsize(dst) / 1e6
print(f"  ✅ model_best.th ({size:.1f} MB)")
print(f"\n✅ Model ready for inference!")

Restoring Stage 1 best model...
  ✅ model_best.th (63.7 MB)

✅ Model ready for inference!


In [8]:
from transformers import AutoModel, AutoTokenizer

print("Pre-downloading TinyBERT...")
AutoTokenizer.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")
AutoModel.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")
print("✅ Done!")

Pre-downloading TinyBERT...


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Done!


model.safetensors:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

In [9]:
import zipfile
import os

zip_path = "/kaggle/working/gector_tinybert_stage1_final.zip"

files_to_zip = [
    # Model weights
    (f"{STAGE1_DIR}/model_best.th", "model_best.th"),
    # Inference scripts
    (f"{work_dir}/gector/gec_model.py", "gec_model.py"),
    (f"{work_dir}/predict.py", "predict.py"),
]

with zipfile.ZipFile(zip_path, "w") as z:
    # Add single files
    for src, arcname in files_to_zip:
        if os.path.exists(src):
            z.write(src, arcname)
            print(f"✅ Added {arcname}")
        else:
            print(f"❌ MISSING: {src}")
    
    # Add entire output_vocabulary folder
    vocab_dir = VOCAB_PATH
    for root, dirs, files in os.walk(vocab_dir):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.join("output_vocabulary", 
                                   os.path.relpath(full_path, vocab_dir))
            z.write(full_path, arcname)
    print(f"✅ Added output_vocabulary/")

size = os.path.getsize(zip_path) / 1e6
print(f"\n✅ Done! gector_tinybert_stage1_final.zip ({size:.1f} MB)")
print(f"Download from: {zip_path}")

✅ Added model_best.th
✅ Added gec_model.py
✅ Added predict.py
✅ Added output_vocabulary/

✅ Done! gector_tinybert_stage1_final.zip (63.8 MB)
Download from: /kaggle/working/gector_tinybert_stage1_final.zip


In [10]:
# Dev set evaluation using ABCN.dev.gold.bea19.m2
dev_orig        = "/kaggle/working/ABCN.dev.orig.txt"
DEV_PRED_OUTPUT = "/kaggle/working/dev_predictions.txt"
DEV_HYP_M2      = "/kaggle/working/dev_hyp.m2"

print("Extracting dev orig from gold m2...")
with open(GOLD_M2, encoding='utf-8') as f_in, \
     open(dev_orig, 'w', encoding='utf-8') as f_out:
    for line in f_in:
        if line.startswith("S "):
            f_out.write(line[2:])
with open(dev_orig) as f:
    count = sum(1 for _ in f)
print(f"✅ Extracted {count} source sentences")

Extracting dev orig from gold m2...
✅ Extracted 4384 source sentences


In [11]:
os.chdir(work_dir)
dev_predict_cmd = f"""python predict.py \
    --model_path {STAGE1_DIR}/model_best.th \
    --vocab_path {VOCAB_PATH} \
    --input_file {dev_orig} \
    --output_file {DEV_PRED_OUTPUT} \
    --transformer_model tinybert \
    --iteration_count 5 \
    --min_error_probability 0.1 \
    --min_probability 0.0 \
    --additional_confidence 0.0 \
    --special_tokens_fix 0 2>&1"""

print("Running inference on dev set...")
output = os.popen(dev_predict_cmd).read()
print(output)

Running inference on dev set...

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 5000.68it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading 

In [12]:
print("Running ERRANT annotation on dev predictions...")
os.system(f"errant_parallel -orig {dev_orig} -cor {DEV_PRED_OUTPUT} -out {DEV_HYP_M2}")

if os.path.exists(DEV_HYP_M2):
    print("✅ dev_hyp.m2 created!")
else:
    print("❌ dev_hyp.m2 not created")
    
# Verify counts match
with open(DEV_HYP_M2) as f:
    hyp_count = sum(1 for line in f if line.startswith("S "))
with open(GOLD_M2) as f:
    gold_count = sum(1 for line in f if line.startswith("S "))
print(f"hyp sentences:  {hyp_count}")
print(f"gold sentences: {gold_count}")

if hyp_count != gold_count:
    print("❌ Sentence count mismatch!")
else:
    print("\n========================================")
    print("ERRANT Scores on BEA-2019 Dev Set:")
    print("(Precision / Recall / F0.5)")
    print("========================================")
    result = os.popen(f"errant_compare -hyp {DEV_HYP_M2} -ref {GOLD_M2} 2>&1").read()
    print(result)

Running ERRANT annotation on dev predictions...
Loading resources...
Processing parallel files...
✅ dev_hyp.m2 created!
hyp sentences:  4384
gold sentences: 4384

ERRANT Scores on BEA-2019 Dev Set:
(Precision / Recall / F0.5)

=========== Span-Based Correction ============
TP	FP	FN	Prec	Rec	F0.5
740	2012	6721	0.2689	0.0992	0.2003




# STAGE 2 DEV RESULT

In [1]:
import os

print("=== Checking all input files ===")
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

=== Checking all input files ===
/kaggle/input/datasets/roseannnnnaguilar/stage2-final-cp/stage2_final_cp/model_best.th
/kaggle/input/datasets/roseannnnnaguilar/gec-test-data/gec_test_data/ABCN.dev.gold.bea19.m2
/kaggle/input/datasets/roseannnnnaguilar/gec-test-data/gec_test_data/ABCN.test.bea19.orig
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/train.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/requirements.txt
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/predict.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/resume_training.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/utils/prepare_clc_fce_data.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/utils/helpers.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/utils/preprocess_data.py
/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/gector/seq2labels_model.py
/kaggle/input/datasets/

In [2]:
import os
import shutil

code_path = "/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code"
work_dir  = "/kaggle/working/gector"

if os.path.exists(work_dir):
    shutil.rmtree(work_dir)
shutil.copytree(code_path, work_dir)
print("✅ Code copied!")

for root, dirs, files in os.walk(work_dir):
    for file in files:
        print(os.path.join(root, file))

✅ Code copied!
/kaggle/working/gector/train.py
/kaggle/working/gector/requirements.txt
/kaggle/working/gector/predict.py
/kaggle/working/gector/resume_training.py
/kaggle/working/gector/utils/preprocess_data.py
/kaggle/working/gector/utils/helpers.py
/kaggle/working/gector/utils/prepare_clc_fce_data.py
/kaggle/working/gector/gector/trainer.py
/kaggle/working/gector/gector/gec_model.py
/kaggle/working/gector/gector/datareader.py
/kaggle/working/gector/gector/seq2labels_model.py
/kaggle/working/gector/data/verb-form-vocab.txt
/kaggle/working/gector/data/output_vocabulary/labels.txt
/kaggle/working/gector/data/output_vocabulary/d_tags.txt
/kaggle/working/gector/data/output_vocabulary/non_padded_namespaces.txt


In [3]:
new_gec_model = '''
import logging
import os
from time import time

import torch
from transformers import AutoTokenizer

from gector.seq2labels_model import Seq2Labels
from utils.helpers import PAD, UNK, get_target_sent_by_edits

logger = logging.getLogger(__file__)

MODEL_PATHS = {
    "bert":          "bert-base-cased",
    "distilbert":    "distilbert-base-uncased",
    "albert":        "albert-base-v2",
    "roberta":       "roberta-base",
    "gpt2":          "gpt2",
    "transformerxl": "transfo-xl-wt103",
    "xlnet":         "xlnet-base-cased",
    "tinybert":      "huawei-noah/TinyBERT_General_4L_312D",
    "mobilebert":    "google/mobilebert-uncased",
}


class SimpleVocab:
    def __init__(self, vocab_path):
        self.token2idx = {}
        self.idx2token = {}
        labels_file = os.path.join(vocab_path, "labels.txt")
        with open(labels_file, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                token = line.strip()
                self.token2idx[token] = idx
                self.idx2token[idx] = token

    def get_token_index(self, token, namespace=None):
        return self.token2idx.get(token, 0)

    def get_token_from_index(self, idx, namespace=None):
        return self.idx2token.get(idx, UNK)

    def __len__(self):
        return len(self.token2idx)


class GecBERTModel(object):
    def __init__(self, vocab_path=None, model_paths=None,
                 weigths=None,
                 max_len=50,
                 min_len=3,
                 lowercase_tokens=False,
                 log=False,
                 iterations=5,
                 min_probability=0.0,
                 model_name="tinybert",
                 special_tokens_fix=0,
                 is_ensemble=False,
                 min_error_probability=0.0,
                 confidence=0.0,
                 resolve_cycles=False):

        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.max_len = max_len
        self.min_len = min_len
        self.lowercase_tokens = lowercase_tokens
        self.min_probability = min_probability
        self.min_error_probability = min_error_probability
        self.log = log
        self.iterations = iterations
        self.confidence = confidence
        self.model_weights = list(map(float, weigths)) if weigths else [1] * len(model_paths)

        self.vocab = SimpleVocab(vocab_path)
        num_labels = len(self.vocab)

        weights_name = MODEL_PATHS.get(model_name, model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(weights_name)

        self.models = []
        for model_path in model_paths:
            model = Seq2Labels(
                encoder_name=weights_name,
                num_labels_classes=num_labels,
                num_detect_classes=4,       
                confidence=confidence,
                special_tokens_fix=0,        
                tune_bert=False,
            ).to(self.device)
            state = torch.load(model_path, map_location=self.device)
            model.load_state_dict(state)
            model.eval()
            self.models.append(model)

        print(f"✅ Loaded {len(self.models)} model(s) on {self.device}")

    def preprocess(self, token_batch):
        seq_lens = [len(s) for s in token_batch if s]
        if not seq_lens:
            return None

        all_input_ids, all_attention_masks, all_offsets = [], [], []

        for sequence in token_batch:
            tokens = sequence[:self.max_len - 1]
            words  = ["$START"] + tokens

            input_ids = []
            offsets   = []
            for word in words:
                word_ids = self.tokenizer.encode(word, add_special_tokens=False)
                if not word_ids:
                    word_ids = [self.tokenizer.unk_token_id]
                offsets.append(len(input_ids))
                input_ids.extend(word_ids)

            cls_id    = self.tokenizer.cls_token_id
            sep_id    = self.tokenizer.sep_token_id
            input_ids = [cls_id] + input_ids + [sep_id]
            offsets   = [o + 1 for o in offsets]

            all_input_ids.append(input_ids)
            all_offsets.append(offsets)

        max_seq_len  = max(len(ids) for ids in all_input_ids)
        max_word_len = max(len(off) for off in all_offsets)
        pad_id = self.tokenizer.pad_token_id or 0

        padded_ids, attn_masks, padded_offsets = [], [], []
        for ids, offsets in zip(all_input_ids, all_offsets):
            pad_len = max_seq_len - len(ids)
            padded_ids.append(ids + [pad_id] * pad_len)
            attn_masks.append([1] * len(ids) + [0] * pad_len)
            off_pad = max_word_len - len(offsets)
            padded_offsets.append(offsets + [0] * off_pad)

        return {
            "input_ids":      torch.tensor(padded_ids,     dtype=torch.long).to(self.device),
            "attention_mask": torch.tensor(attn_masks,     dtype=torch.long).to(self.device),
            "offsets":        torch.tensor(padded_offsets, dtype=torch.long).to(self.device),
        }

    def predict(self, batch):
        t0 = time()
        predictions = []
        for model in self.models:
            with torch.no_grad():
                output = model.forward(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    offsets=batch["offsets"],
                )
            predictions.append(output)
        probs, idx, error_probs = self._convert(predictions)
        if self.log:
            print(f"Inference time {time() - t0:.2f}s")
        return probs, idx, error_probs

    def _convert(self, data):
        all_class_probs = torch.zeros_like(data[0]["class_probabilities_labels"])
        error_probs     = torch.zeros_like(data[0]["max_error_probability"])
        for output, weight in zip(data, self.model_weights):
            all_class_probs += weight * output["class_probabilities_labels"] / sum(self.model_weights)
            error_probs     += weight * output["max_error_probability"]      / sum(self.model_weights)
        max_vals = torch.max(all_class_probs, dim=-1)
        return max_vals[0].tolist(), max_vals[1].tolist(), error_probs.tolist()

    def get_token_action(self, token, index, prob, sugg_token):
        if prob < self.min_probability or sugg_token in [UNK, PAD, "$KEEP"]:
            return None
        if sugg_token.startswith("$REPLACE_") or sugg_token.startswith("$TRANSFORM_") or sugg_token == "$DELETE":
            start_pos, end_pos = index, index + 1
        elif sugg_token.startswith("$APPEND_") or sugg_token.startswith("$MERGE_"):
            start_pos = end_pos = index + 1
        else:
            return None
        sugg_token_clear = "" if sugg_token == "$DELETE" else (
            sugg_token if sugg_token.startswith("$TRANSFORM_") or sugg_token.startswith("$MERGE_")
            else sugg_token[sugg_token.index("_") + 1:]
        )
        return start_pos - 1, end_pos - 1, sugg_token_clear, prob

    def postprocess_batch(self, batch, all_probabilities, all_idxs, error_probs):
        all_results = []
        noop_index = self.vocab.get_token_index("$KEEP", "labels")
        for tokens, probabilities, idxs, error_prob in zip(
                batch, all_probabilities, all_idxs, error_probs):
            length = min(len(tokens), self.max_len)
            if max(idxs) == 0 or error_prob < self.min_error_probability:
                all_results.append(tokens)
                continue
            edits = []
            for i in range(length):
                token = tokens[i - 1]
                if idxs[i] == noop_index:
                    continue
                sugg_token = self.vocab.get_token_from_index(idxs[i], "labels")
                action = self.get_token_action(token, i, probabilities[i], sugg_token)
                if action:
                    edits.append(action)
            all_results.append(get_target_sent_by_edits(tokens, edits))
        return all_results

    def update_final_batch(self, final_batch, pred_ids, pred_batch, prev_preds_dict):
        new_pred_ids  = []
        total_updated = 0
        for i, orig_id in enumerate(pred_ids):
            orig       = final_batch[orig_id]
            pred       = pred_batch[i]
            prev_preds = prev_preds_dict[orig_id]
            if orig != pred and pred not in prev_preds:
                final_batch[orig_id] = pred
                new_pred_ids.append(orig_id)
                prev_preds_dict[orig_id].append(pred)
                total_updated += 1
            elif orig != pred and pred in prev_preds:
                final_batch[orig_id] = pred
                total_updated += 1
        return final_batch, new_pred_ids, total_updated

    def handle_batch(self, full_batch):
        final_batch     = full_batch[:]
        batch_size      = len(full_batch)
        prev_preds_dict = {i: [final_batch[i]] for i in range(batch_size)}
        short_ids       = [i for i in range(batch_size) if len(full_batch[i]) < self.min_len]
        pred_ids        = [i for i in range(batch_size) if i not in short_ids]
        total_updates   = 0

        for n_iter in range(self.iterations):
            orig_batch = [final_batch[i] for i in pred_ids]
            sequences  = self.preprocess(orig_batch)
            if not sequences:
                break
            probabilities, idxs, error_probs = self.predict(sequences)
            pred_batch = self.postprocess_batch(orig_batch, probabilities, idxs, error_probs)
            if self.log:
                print(f"Iteration {n_iter + 1}: {round(100*len(pred_ids)/batch_size, 1)}% of sentences active")
            final_batch, pred_ids, cnt = self.update_final_batch(
                final_batch, pred_ids, pred_batch, prev_preds_dict)
            total_updates += cnt
            if not pred_ids:
                break

        return final_batch, total_updates
'''

import os
with open(f"{work_dir}/gector/gec_model.py", "w") as f:
    f.write(new_gec_model)
print("✅ gec_model.py!")

✅ gec_model.py!


In [4]:
content = open(f"{work_dir}/predict.py").read()
content = content.replace(
    "choices=['bert', 'gpt2', 'transformerxl', 'xlnet', 'distilbert', 'roberta', 'albert'],",
    ""
)
content = content.replace(
    "default='roberta'",
    "default='tinybert'"
)
with open(f"{work_dir}/predict.py", "w") as f:
    f.write(content)
print("✅ predict.py updated!")

✅ predict.py updated!


In [5]:
import os
os.system("pip install transformers torch tqdm python-Levenshtein -q")
os.system("pip install errant -q")
print("✅ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.3/499.3 kB 9.3 MB/s eta 0:00:00
✅ Dependencies installed!


In [6]:
import os

data_path = "/kaggle/input/datasets/roseannnnnaguilar/gec-gector-datasets-new/gec_gector_datasets_new"
test_path = "/kaggle/input/datasets/roseannnnnaguilar/gec-test-data/gec_test_data"
work_dir  = "/kaggle/working/gector"

VOCAB_PATH   = "/kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/data/output_vocabulary"
GOLD_M2      = f"{test_path}/ABCN.dev.gold.bea19.m2"

STAGE2_DIR   = "/kaggle/working/models/stage2"
PRED_OUTPUT  = "/kaggle/working/predictions.txt"
HYP_M2       = "/kaggle/working/hyp.m2"

os.makedirs(STAGE2_DIR, exist_ok=True)

print("✅ Paths defined!")
print("\nVerifying files:")
for label, path in [
    ("Vocab",    VOCAB_PATH),
    ("Gold M2",  GOLD_M2),
]:
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"  {status} {label}: {path}")


✅ Paths defined!

Verifying files:
  ✅ Vocab: /kaggle/input/datasets/roseannnnnaguilar/gector-code/gector_code/data/output_vocabulary
  ✅ Gold M2: /kaggle/input/datasets/roseannnnnaguilar/gec-test-data/gec_test_data/ABCN.dev.gold.bea19.m2


In [8]:
import shutil
import os


stage2_checkpoint = "/kaggle/input/datasets/roseannnnnaguilar/stage2-final-cp/stage2_final_cp"

print("Restoring Stage 2 best model...")
src = os.path.join(stage2_checkpoint, "model_best.th")   
dst = os.path.join(STAGE2_DIR, "model_best.th")
shutil.copy(src, dst)
size = os.path.getsize(dst) / 1e6
print(f"  ✅ model_best.th ({size:.1f} MB)")
print(f"\n✅ Model ready for inference!")

Restoring Stage 2 best model...
  ✅ model_best.th (63.7 MB)

✅ Model ready for inference!


In [9]:
import zipfile
import os

zip_path = "/kaggle/working/gector_tinybert_stage2_final.zip"

files_to_zip = [
    # Model weights
    (f"{STAGE2_DIR}/model_best.th", "model_best.th"),
    # Inference scripts
    (f"{work_dir}/gector/gec_model.py", "gec_model.py"),
    (f"{work_dir}/predict.py", "predict.py"),
]

with zipfile.ZipFile(zip_path, "w") as z:
    # Add single files
    for src, arcname in files_to_zip:
        if os.path.exists(src):
            z.write(src, arcname)
            print(f"✅ Added {arcname}")
        else:
            print(f"❌ MISSING: {src}")
    
    # Add entire output_vocabulary folder
    vocab_dir = VOCAB_PATH
    for root, dirs, files in os.walk(vocab_dir):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.join("output_vocabulary", 
                                   os.path.relpath(full_path, vocab_dir))
            z.write(full_path, arcname)
    print(f"✅ Added output_vocabulary/")

size = os.path.getsize(zip_path) / 1e6
print(f"\n✅ Done! gector_tinybert_stage2_final.zip ({size:.1f} MB)")
print(f"Download from: {zip_path}")

✅ Added model_best.th
✅ Added gec_model.py
✅ Added predict.py
✅ Added output_vocabulary/

✅ Done! gector_tinybert_stage2_final.zip (63.8 MB)
Download from: /kaggle/working/gector_tinybert_stage2_final.zip


In [10]:
# Dev set evaluation using ABCN.dev.gold.bea19.m2
dev_orig        = "/kaggle/working/ABCN.dev.orig.txt"
DEV_PRED_OUTPUT = "/kaggle/working/dev_predictions.txt"
DEV_HYP_M2      = "/kaggle/working/dev_hyp.m2"

print("Extracting dev orig from gold m2...")
with open(GOLD_M2, encoding='utf-8') as f_in, \
     open(dev_orig, 'w', encoding='utf-8') as f_out:
    for line in f_in:
        if line.startswith("S "):
            f_out.write(line[2:])
with open(dev_orig) as f:
    count = sum(1 for _ in f)
print(f"✅ Extracted {count} source sentences")

Extracting dev orig from gold m2...
✅ Extracted 4384 source sentences


In [11]:
os.chdir(work_dir)
dev_predict_cmd = f"""python predict.py \
    --model_path {STAGE2_DIR}/model_best.th \
    --vocab_path {VOCAB_PATH} \
    --input_file {dev_orig} \
    --output_file {DEV_PRED_OUTPUT} \
    --transformer_model tinybert \
    --iteration_count 5 \
    --min_error_probability 0.1 \
    --min_probability 0.0 \
    --additional_confidence 0.0 \
    --special_tokens_fix 0 2>&1"""

print("Running inference on dev set...")
output = os.popen(dev_predict_cmd).read()
print(output)

Running inference on dev set...

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 4541.92it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading 

In [12]:
print("Running ERRANT annotation on dev predictions...")
os.system(f"errant_parallel -orig {dev_orig} -cor {DEV_PRED_OUTPUT} -out {DEV_HYP_M2}")

if os.path.exists(DEV_HYP_M2):
    print("✅ dev_hyp.m2 created!")
else:
    print("❌ dev_hyp.m2 not created")
    
# Verify counts match
with open(DEV_HYP_M2) as f:
    hyp_count = sum(1 for line in f if line.startswith("S "))
with open(GOLD_M2) as f:
    gold_count = sum(1 for line in f if line.startswith("S "))
print(f"hyp sentences:  {hyp_count}")
print(f"gold sentences: {gold_count}")

if hyp_count != gold_count:
    print("❌ Sentence count mismatch!")
else:
    print("\n========================================")
    print("ERRANT Scores on BEA-2019 Dev Set:")
    print("(Precision / Recall / F0.5)")
    print("========================================")
    result = os.popen(f"errant_compare -hyp {DEV_HYP_M2} -ref {GOLD_M2} 2>&1").read()
    print(result)

Running ERRANT annotation on dev predictions...
Loading resources...
Processing parallel files...
✅ dev_hyp.m2 created!
hyp sentences:  4384
gold sentences: 4384

ERRANT Scores on BEA-2019 Dev Set:
(Precision / Recall / F0.5)

=========== Span-Based Correction ============
TP	FP	FN	Prec	Rec	F0.5
273	181	7188	0.6013	0.0366	0.1471




# STAGE 3 DEV RESULT

In [1]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# ── Cell 2: Verify GPU ──────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# ── Cell 3: Paths — UPDATE DRIVE_ROOT to match your Shared Drive ──────────────
import os

# ── EDIT THESE ──
DRIVE_ROOT   = "/content/drive/Shareddrives/gec_thesis/gec_thesis"
DATASETS_DIR = f"{DRIVE_ROOT}/gec_gector_datasets_new"
TEST_DIR     = f"{DRIVE_ROOT}/gec_test_data"
CODE_DIR     = f"{DRIVE_ROOT}/gector_code"
STAGE3_CKPT  = f"{DRIVE_ROOT}/models/stage3/tinybert"   # folder with model_best.th
EVAL_OUT     = f"{DRIVE_ROOT}/models/eval/tinybert"      # evaluation outputs go here

# ── Local working dirs ──
WORK_DIR   = "/content/gector"
STAGE3_DIR = "/content/models/stage3/tinybert"

for d in [WORK_DIR, STAGE3_DIR, EVAL_OUT]:
    os.makedirs(d, exist_ok=True)

# ── Dataset paths ──
WI_TRAIN   = f"{DATASETS_DIR}/wi_train_gector.txt"
WI_DEV     = f"{DATASETS_DIR}/wi_dev_gector.txt"
VOCAB_PATH = f"{WORK_DIR}/data/output_vocabulary"

# ── Eval paths ──
GOLD_M2     = f"{TEST_DIR}/ABCN.dev.gold.bea19.m2"
DEV_ORIG    = "/content/ABCN.dev.orig.txt"
DEV_PRED    = "/content/dev_predictions.txt"
DEV_HYP_M2  = "/content/dev_hyp.m2"

print("✅ Paths defined!")
print(f"   DRIVE_ROOT  : {DRIVE_ROOT}")
print(f"   CODE_DIR    : {CODE_DIR}")
print(f"   STAGE3_CKPT : {STAGE3_CKPT}")
print(f"   EVAL_OUT    : {EVAL_OUT}")


✅ Paths defined!
   DRIVE_ROOT  : /content/drive/Shareddrives/gec_thesis/gec_thesis
   CODE_DIR    : /content/drive/Shareddrives/gec_thesis/gec_thesis/gector_code
   STAGE3_CKPT : /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage3/tinybert
   EVAL_OUT    : /content/drive/Shareddrives/gec_thesis/gec_thesis/models/eval/tinybert


In [4]:
# ── Cell 4: Verify all input files ────────────────────────────────────────────
import os

checks = [
    ("W&I Train",                      WI_TRAIN),
    ("W&I Dev",                        WI_DEV),
    ("Gold M2",                        GOLD_M2),
    ("Stage3 model_best.th",           os.path.join(STAGE3_CKPT, "model_best.th")),
    ("Code dir",                       CODE_DIR),
]

all_ok = True
for name, path in checks:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {name}: {path}")
    if not exists:
        all_ok = False

print()
print("✅ All files found — ready to proceed!" if all_ok else "❌ Fix missing files above before continuing.")


  ✅ W&I Train: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_train_gector.txt
  ✅ W&I Dev: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_gector_datasets_new/wi_dev_gector.txt
  ✅ Gold M2: /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_test_data/ABCN.dev.gold.bea19.m2
  ✅ Stage3 model_best.th: /content/drive/Shareddrives/gec_thesis/gec_thesis/models/stage3/tinybert/model_best.th
  ✅ Code dir: /content/drive/Shareddrives/gec_thesis/gec_thesis/gector_code

✅ All files found — ready to proceed!


In [5]:
# ── Cell 5: Copy gector_code_tinybert to local working dir ──────────────────────
import shutil, os

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(CODE_DIR, WORK_DIR)
print("✅ Code copied to", WORK_DIR)

for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        print(os.path.join(root, f).replace(WORK_DIR, ""))


✅ Code copied to /content/gector
/predict.py
/resume_training.py
/requirements.txt
/train.py
/gector/trainer.py
/gector/datareader.py
/gector/seq2labels_model.py
/gector/gec_model.py
/data/verb-form-vocab.txt
/data/output_vocabulary/non_padded_namespaces.txt
/data/output_vocabulary/labels.txt
/data/output_vocabulary/d_tags.txt
/utils/preprocess_data.py
/utils/helpers.py
/utils/prepare_clc_fce_data.py


In [6]:
# ── Cell 6: Patch gec_model.py for TINYBERT ─────────────────────────────────────
new_gec_model = '''
import logging
import os
from time import time

import torch
from transformers import AutoTokenizer

from gector.seq2labels_model import Seq2Labels
from utils.helpers import PAD, UNK, get_target_sent_by_edits

logger = logging.getLogger(__file__)

MODEL_PATHS = {
    "bert":          "bert-base-cased",
    "distilbert":    "distilbert-base-uncased",
    "albert":        "albert-base-v2",
    "roberta":       "roberta-base",
    "gpt2":          "gpt2",
    "transformerxl": "transfo-xl-wt103",
    "xlnet":         "xlnet-base-cased",
    "tinybert":      "huawei-noah/TinyBERT_General_4L_312D",
    "mobilebert":    "google/mobilebert-uncased",
}


class SimpleVocab:
    def __init__(self, vocab_path):
        self.token2idx = {}
        self.idx2token = {}
        labels_file = os.path.join(vocab_path, "labels.txt")
        with open(labels_file, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                token = line.strip()
                self.token2idx[token] = idx
                self.idx2token[idx] = token

    def get_token_index(self, token, namespace=None):
        return self.token2idx.get(token, 0)

    def get_token_from_index(self, idx, namespace=None):
        return self.idx2token.get(idx, UNK)

    def __len__(self):
        return len(self.token2idx)


class GecBERTModel(object):
    def __init__(self, vocab_path=None, model_paths=None,
                 weigths=None,
                 max_len=50,
                 min_len=3,
                 lowercase_tokens=False,
                 log=False,
                 iterations=5,
                 min_probability=0.0,
                 model_name="tinybert",
                 special_tokens_fix=0,
                 is_ensemble=False,
                 min_error_probability=0.0,
                 confidence=0.0,
                 resolve_cycles=False):

        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.max_len = max_len
        self.min_len = min_len
        self.lowercase_tokens = lowercase_tokens
        self.min_probability = min_probability
        self.min_error_probability = min_error_probability
        self.log = log
        self.iterations = iterations
        self.confidence = confidence
        self.model_weights = list(map(float, weigths)) if weigths else [1] * len(model_paths)

        self.vocab = SimpleVocab(vocab_path)
        num_labels = len(self.vocab)

        weights_name = MODEL_PATHS.get(model_name, model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(weights_name)

        self.models = []
        for model_path in model_paths:
            model = Seq2Labels(
                encoder_name=weights_name,
                num_labels_classes=num_labels,
                num_detect_classes=4,
                confidence=confidence,
                special_tokens_fix=0,
                tune_bert=False,
            ).to(self.device)
            state = torch.load(model_path, map_location=self.device)
            model.load_state_dict(state)
            model.eval()
            self.models.append(model)

        print(f"✅ Loaded {len(self.models)} model(s) on {self.device}")

    def preprocess(self, token_batch):
        seq_lens = [len(s) for s in token_batch if s]
        if not seq_lens:
            return None

        all_input_ids, all_attention_masks, all_offsets = [], [], []

        for sequence in token_batch:
            tokens = sequence[:self.max_len - 1]
            words  = ["$START"] + tokens

            input_ids = []
            offsets   = []
            for word in words:
                word_ids = self.tokenizer.encode(word, add_special_tokens=False)
                if not word_ids:
                    word_ids = [self.tokenizer.unk_token_id]
                offsets.append(len(input_ids))
                input_ids.extend(word_ids)

            cls_id    = self.tokenizer.cls_token_id
            sep_id    = self.tokenizer.sep_token_id
            input_ids = [cls_id] + input_ids + [sep_id]
            offsets   = [o + 1 for o in offsets]

            all_input_ids.append(input_ids)
            all_offsets.append(offsets)

        max_seq_len  = max(len(ids) for ids in all_input_ids)
        max_word_len = max(len(off) for off in all_offsets)
        pad_id = self.tokenizer.pad_token_id or 0

        padded_ids, attn_masks, padded_offsets = [], [], []
        for ids, offsets in zip(all_input_ids, all_offsets):
            pad_len = max_seq_len - len(ids)
            padded_ids.append(ids + [pad_id] * pad_len)
            attn_masks.append([1] * len(ids) + [0] * pad_len)
            off_pad = max_word_len - len(offsets)
            padded_offsets.append(offsets + [0] * off_pad)

        return {
            "input_ids":      torch.tensor(padded_ids,     dtype=torch.long).to(self.device),
            "attention_mask": torch.tensor(attn_masks,     dtype=torch.long).to(self.device),
            "offsets":        torch.tensor(padded_offsets, dtype=torch.long).to(self.device),
        }

    def predict(self, batch):
        t0 = time()
        predictions = []
        for model in self.models:
            with torch.no_grad():
                output = model.forward(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    offsets=batch["offsets"],
                )
            predictions.append(output)
        probs, idx, error_probs = self._convert(predictions)
        if self.log:
            print(f"Inference time {time() - t0:.2f}s")
        return probs, idx, error_probs

    def _convert(self, data):
        all_class_probs = torch.zeros_like(data[0]["class_probabilities_labels"])
        error_probs     = torch.zeros_like(data[0]["max_error_probability"])
        for output, weight in zip(data, self.model_weights):
            all_class_probs += weight * output["class_probabilities_labels"] / sum(self.model_weights)
            error_probs     += weight * output["max_error_probability"]      / sum(self.model_weights)
        max_vals = torch.max(all_class_probs, dim=-1)
        return max_vals[0].tolist(), max_vals[1].tolist(), error_probs.tolist()

    def get_token_action(self, token, index, prob, sugg_token):
        if prob < self.min_probability or sugg_token in [UNK, PAD, "$KEEP"]:
            return None
        if sugg_token.startswith("$REPLACE_") or sugg_token.startswith("$TRANSFORM_") or sugg_token == "$DELETE":
            start_pos, end_pos = index, index + 1
        elif sugg_token.startswith("$APPEND_") or sugg_token.startswith("$MERGE_"):
            start_pos = end_pos = index + 1
        else:
            return None
        sugg_token_clear = "" if sugg_token == "$DELETE" else (
            sugg_token if sugg_token.startswith("$TRANSFORM_") or sugg_token.startswith("$MERGE_")
            else sugg_token[sugg_token.index("_") + 1:]
        )
        return start_pos - 1, end_pos - 1, sugg_token_clear, prob

    def postprocess_batch(self, batch, all_probabilities, all_idxs, error_probs):
        all_results = []
        noop_index = self.vocab.get_token_index("$KEEP", "labels")
        for tokens, probabilities, idxs, error_prob in zip(
                batch, all_probabilities, all_idxs, error_probs):
            length = min(len(tokens), self.max_len)
            if max(idxs) == 0 or error_prob < self.min_error_probability:
                all_results.append(tokens)
                continue
            edits = []
            for i in range(length):
                token = tokens[i - 1]
                if idxs[i] == noop_index:
                    continue
                sugg_token = self.vocab.get_token_from_index(idxs[i], "labels")
                action = self.get_token_action(token, i, probabilities[i], sugg_token)
                if action:
                    edits.append(action)
            all_results.append(get_target_sent_by_edits(tokens, edits))
        return all_results

    def update_final_batch(self, final_batch, pred_ids, pred_batch, prev_preds_dict):
        new_pred_ids  = []
        total_updated = 0
        for i, orig_id in enumerate(pred_ids):
            orig       = final_batch[orig_id]
            pred       = pred_batch[i]
            prev_preds = prev_preds_dict[orig_id]
            if orig != pred and pred not in prev_preds:
                final_batch[orig_id] = pred
                new_pred_ids.append(orig_id)
                prev_preds_dict[orig_id].append(pred)
                total_updated += 1
            elif orig != pred and pred in prev_preds:
                final_batch[orig_id] = pred
                total_updated += 1
        return final_batch, new_pred_ids, total_updated

    def handle_batch(self, full_batch):
        final_batch     = full_batch[:]
        batch_size      = len(full_batch)
        prev_preds_dict = {i: [final_batch[i]] for i in range(batch_size)}
        short_ids       = [i for i in range(batch_size) if len(full_batch[i]) < self.min_len]
        pred_ids        = [i for i in range(batch_size) if i not in short_ids]
        total_updates   = 0

        for n_iter in range(self.iterations):
            orig_batch = [final_batch[i] for i in pred_ids]
            sequences  = self.preprocess(orig_batch)
            if not sequences:
                break
            probabilities, idxs, error_probs = self.predict(sequences)
            pred_batch = self.postprocess_batch(orig_batch, probabilities, idxs, error_probs)
            if self.log:
                print(f"Iteration {n_iter + 1}: {round(100*len(pred_ids)/batch_size, 1)}% of sentences active")
            final_batch, pred_ids, cnt = self.update_final_batch(
                final_batch, pred_ids, pred_batch, prev_preds_dict)
            total_updates += cnt
            if not pred_ids:
                break

        return final_batch, total_updates
'''

import os
with open(f"{WORK_DIR}/gector/gec_model.py", "w") as f:
    f.write(new_gec_model)
print("✅ gec_model.py patched for TINYBERT!")


✅ gec_model.py patched for TINYBERT!


In [7]:
# ── Cell 7: Restore Stage 3 checkpoint from Drive ────────────────────────────────
import shutil, os

os.makedirs(STAGE3_DIR, exist_ok=True)

print("Restoring Stage 3 best model from Drive...")
src = os.path.join(STAGE3_CKPT, "model_best.th")
dst = os.path.join(STAGE3_DIR, "model_best.th")
if os.path.exists(src):
    shutil.copy(src, dst)
    size = os.path.getsize(dst) / 1e6
    print(f"  ✅ model_best.th ({size:.1f} MB)")
else:
    print(f"  ❌ Not found: {src}")

print("\n✅ Model ready for inference!")


Restoring Stage 3 best model from Drive...
  ✅ model_best.th (63.7 MB)

✅ Model ready for inference!


In [8]:
# ── Cell 8: Pre-download TinyBERT weights ─────────────────────────────────────
from transformers import AutoModel, AutoTokenizer

print("Pre-downloading TinyBERT weights from HuggingFace...")
AutoTokenizer.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")
AutoModel.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")
print("✅ TinyBERT weights ready!")


Pre-downloading TinyBERT weights from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ TinyBERT weights ready!


In [9]:
content = open(f"{WORK_DIR}/predict.py").read()

content = content.replace(
    "choices=['bert', 'gpt2', 'transformerxl', 'xlnet', 'distilbert', 'roberta', 'albert']",
    "choices=['bert', 'gpt2', 'transformerxl', 'xlnet', 'distilbert', 'roberta', 'albert', 'tinybert']"
)

with open(f"{WORK_DIR}/predict.py", "w") as f:
    f.write(content)

print("✅ TinyBERT added to predict.py!")

✅ TinyBERT added to predict.py!


In [10]:
# ── Cell 9: BEA-2019 Test Set Inference + submission.zip ──────────────────────
import shutil, os, subprocess, sys, zipfile, time, torch

# ── Paths ──
TEST_ORIG   = f"{TEST_DIR}/ABCN.test.bea19.orig"
PRED_OUTPUT = "/content/predictions.txt"

with open(TEST_ORIG) as f:
    count = sum(1 for _ in f)
print(f"✅ Test input ready ({count} sentences) → {TEST_ORIG}")

# ── GPU RAM before inference ──────────────────────────────────────────────────
print()
print("=" * 50)
print("GPU RAM BEFORE TEST INFERENCE")
print("=" * 50)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(0)
    mem_before = torch.cuda.memory_allocated(0) / 1e6
    print(f"  Allocated : {mem_before:.1f} MB")
    print(f"  Reserved  : {torch.cuda.memory_reserved(0) / 1e6:.1f} MB")

# ── Model parameters ──────────────────────────────────────────────────────────
print()
print("=" * 50)
print("MODEL PARAMETERS")
print("=" * 50)
try:
    _state = torch.load(f"{STAGE3_DIR}/model_best.th", map_location="cpu")
    _params = sum(p.numel() for p in _state.values() if hasattr(p, "numel"))
    _size   = sum(p.numel() * p.element_size() for p in _state.values() if hasattr(p, "numel"))
    print(f"  Total parameters : {_params:,}  ({_params/1e6:.2f} M)")
    print(f"  Model size       : {_size / 1e6:.1f} MB (float32)")
    del _state
except Exception as e:
    print(f"  Could not load state dict: {e}")

# ── Run inference ──
os.chdir(WORK_DIR)
predict_cmd = [
    "python", "predict.py",
    "--model_path",            f"{STAGE3_DIR}/model_best.th",
    "--vocab_path",            VOCAB_PATH,
    "--input_file",            TEST_ORIG,
    "--output_file",           PRED_OUTPUT,
    "--transformer_model",     "tinybert",
    "--iteration_count",       "5",
    "--min_error_probability", "0.1",
    "--min_probability",       "0.0",
    "--additional_confidence", "0.0",
    "--special_tokens_fix",    "0",
]

print()
print("Running inference on BEA-2019 test set (TINYBERT)...")
t_start = time.time()
process = subprocess.Popen(
    predict_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()
t_end = time.time()

with open(PRED_OUTPUT) as f:
    pred_count = sum(1 for _ in f)
print(f"\n✅ Predictions written ({pred_count} lines) → {PRED_OUTPUT}")

# ── Inference time + GPU RAM after ───────────────────────────────────────────
elapsed  = t_end - t_start
per_sent = elapsed / count if count > 0 else 0

print()
print("=" * 50)
print("INFERENCE STATS — TEST SET")
print("=" * 50)
print(f"  Sentences        : {count}")
print(f"  Total time       : {elapsed:.1f} s  ({elapsed/60:.2f} min)")
print(f"  Time / sentence  : {per_sent*1000:.1f} ms")
if torch.cuda.is_available():
    mem_after = torch.cuda.memory_allocated(0) / 1e6
    mem_peak  = torch.cuda.max_memory_allocated(0) / 1e6
    print(f"  GPU RAM before   : {mem_before:.1f} MB")
    print(f"  GPU RAM after    : {mem_after:.1f} MB")
    print(f"  GPU RAM peak     : {mem_peak:.1f} MB")
    print(f"  GPU RAM delta    : {mem_after - mem_before:+.1f} MB")

# ── nvidia-smi snapshot ───────────────────────────────────────────────────────
print()
print("=" * 50)
print("GPU USAGE (nvidia-smi)")
print("=" * 50)
_smi = subprocess.run(
    ["nvidia-smi",
     "--query-gpu=index,name,temperature.gpu,utilization.gpu,utilization.memory,memory.used,memory.free,memory.total",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
if _smi.returncode == 0:
    for line in _smi.stdout.strip().split("\n"):
        idx, name, temp, gpu_util, mem_util, mem_used, mem_free, mem_total = [x.strip() for x in line.split(",")]
        print(f"  GPU {idx}        : {name}")
        print(f"  Temperature  : {temp} °C")
        print(f"  GPU Util     : {gpu_util} %")
        print(f"  Memory Util  : {mem_util} %")
        print(f"  Memory Used  : {mem_used} MB / {mem_total} MB")
        print(f"  Memory Free  : {mem_free} MB")
else:
    print("  nvidia-smi not available:", _smi.stderr)

print()
print("=" * 50)
print("TORCH CUDA MEMORY")
print("=" * 50)
print(f"  Allocated : {round(torch.cuda.memory_allocated(0) / 1e6, 1)} MB")
print(f"  Reserved  : {round(torch.cuda.memory_reserved(0) / 1e6, 1)} MB")

# ── Package submission.zip ──
zip_path = "/content/submission.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    z.write(PRED_OUTPUT, "predictions.txt")
size = os.path.getsize(zip_path) / 1024
print(f"\n✅ submission.zip ready ({size:.1f} KB) → {zip_path}")
print("Download submission.zip and upload to CodaBench!")

# ── Also save to Drive ──
drive_pred = os.path.join(EVAL_OUT, "test_predictions.txt")
drive_zip  = os.path.join(EVAL_OUT, "submission.zip")
shutil.copy(PRED_OUTPUT, drive_pred)
shutil.copy(zip_path,    drive_zip)
print(f"✅ Also saved to Drive: {drive_zip}")
print(f"   → {EVAL_OUT}")

✅ Test input ready (4477 sentences) → /content/drive/Shareddrives/gec_thesis/gec_thesis/gec_test_data/ABCN.test.bea19.orig

GPU RAM BEFORE TEST INFERENCE
  Allocated : 0.0 MB
  Reserved  : 0.0 MB

MODEL PARAMETERS
  Total parameters : 15,917,126  (15.92 M)
  Model size       : 63.7 MB (float32)

Running inference on BEA-2019 test set (TINYBERT)...

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 3032.14it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.seq_relationship.weight               

In [11]:
# ── Cell 10: Extract dev orig sentences from Gold M2 ──────────────────────────
import os

print("Extracting dev orig from gold m2...")
with open(GOLD_M2, encoding='utf-8') as f_in, \
     open(DEV_ORIG, 'w', encoding='utf-8') as f_out:
    for line in f_in:
        if line.startswith("S "):
            f_out.write(line[2:])

with open(DEV_ORIG) as f:
    count = sum(1 for _ in f)
print(f"✅ Extracted {count} source sentences → {DEV_ORIG}")


Extracting dev orig from gold m2...
✅ Extracted 4384 source sentences → /content/ABCN.dev.orig.txt


In [12]:
# ── Cell 11: Run inference on dev set ─────────────────────────────────────────
import subprocess, os, sys, time, torch

os.chdir(WORK_DIR)

predict_cmd = [
    "python", "predict.py",
    "--model_path",           f"{STAGE3_DIR}/model_best.th",
    "--vocab_path",           VOCAB_PATH,
    "--input_file",           DEV_ORIG,
    "--output_file",          DEV_PRED,
    "--transformer_model",    "tinybert",
    "--iteration_count",      "5",
    "--min_error_probability","0.1",
    "--min_probability",      "0.0",
    "--additional_confidence","0.0",
    "--special_tokens_fix",   "0",
]

# ── GPU RAM before inference ──────────────────────────────────────────────────
print("=" * 50)
print("GPU RAM BEFORE DEV INFERENCE")
print("=" * 50)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(0)
    mem_before_dev = torch.cuda.memory_allocated(0) / 1e6
    print(f"  Allocated : {mem_before_dev:.1f} MB")
    print(f"  Reserved  : {torch.cuda.memory_reserved(0) / 1e6:.1f} MB")

# ── Model parameters ──────────────────────────────────────────────────────────
print("=" * 50)
print("MODEL PARAMETERS")
print("=" * 50)
try:
    _state = torch.load(f"{STAGE3_DIR}/model_best.th", map_location="cpu")
    _params = sum(p.numel() for p in _state.values() if hasattr(p, "numel"))
    print(f"  Total parameters : {_params:,}  ({_params/1e6:.2f} M)")
    del _state
except Exception as e:
    print(f"  Could not load state dict: {e}")
print()

print("Running inference on dev set (TINYBERT)...")
print(f"  Input : {DEV_ORIG}")
print(f"  Output: {DEV_PRED}")
t_start = time.time()
process = subprocess.Popen(
    predict_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
t_end = time.time()
print("\n✅ Inference done!")

# ── Count sentences ───────────────────────────────────────────────────────────
with open(DEV_ORIG) as f:
    dev_count = sum(1 for _ in f)

elapsed_dev  = t_end - t_start
per_sent_dev = elapsed_dev / dev_count if dev_count > 0 else 0

# ── Inference time + GPU RAM after ───────────────────────────────────────────
print()
print("=" * 50)
print("INFERENCE STATS — DEV SET")
print("=" * 50)
print(f"  Sentences        : {dev_count}")
print(f"  Total time       : {elapsed_dev:.1f} s  ({elapsed_dev/60:.2f} min)")
print(f"  Time / sentence  : {per_sent_dev*1000:.1f} ms")
if torch.cuda.is_available():
    mem_after_dev = torch.cuda.memory_allocated(0) / 1e6
    mem_peak_dev  = torch.cuda.max_memory_allocated(0) / 1e6
    print(f"  GPU RAM before   : {mem_before_dev:.1f} MB")
    print(f"  GPU RAM after    : {mem_after_dev:.1f} MB")
    print(f"  GPU RAM peak     : {mem_peak_dev:.1f} MB")
    print(f"  GPU RAM delta    : {mem_after_dev - mem_before_dev:+.1f} MB")

# ── nvidia-smi snapshot ───────────────────────────────────────────────────────
print()
print("=" * 50)
print("GPU USAGE (nvidia-smi)")
print("=" * 50)
_smi = subprocess.run(
    ["nvidia-smi",
     "--query-gpu=index,name,temperature.gpu,utilization.gpu,utilization.memory,memory.used,memory.free,memory.total",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
if _smi.returncode == 0:
    for line in _smi.stdout.strip().split("\n"):
        idx, name, temp, gpu_util, mem_util, mem_used, mem_free, mem_total = [x.strip() for x in line.split(",")]
        print(f"  GPU {idx}        : {name}")
        print(f"  Temperature  : {temp} °C")
        print(f"  GPU Util     : {gpu_util} %")
        print(f"  Memory Util  : {mem_util} %")
        print(f"  Memory Used  : {mem_used} MB / {mem_total} MB")
        print(f"  Memory Free  : {mem_free} MB")
else:
    print("  nvidia-smi not available:", _smi.stderr)

print()
print("=" * 50)
print("TORCH CUDA MEMORY")
print("=" * 50)
print(f"  Allocated : {round(torch.cuda.memory_allocated(0) / 1e6, 1)} MB")
print(f"  Reserved  : {round(torch.cuda.memory_reserved(0) / 1e6, 1)} MB")

# ── Save dev predictions to Drive ──
import shutil
shutil.copy(DEV_PRED, os.path.join(EVAL_OUT, "dev_predictions.txt"))
print(f"\n✅ dev_predictions.txt saved to Drive: {EVAL_OUT}")


GPU RAM BEFORE DEV INFERENCE
  Allocated : 0.0 MB
  Reserved  : 0.0 MB
MODEL PARAMETERS
  Total parameters : 15,917,126  (15.92 M)

Running inference on dev set (TINYBERT)...
  Input : /content/ABCN.dev.orig.txt
  Output: /content/dev_predictions.txt

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 2548.51it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.d

In [13]:
import os
os.system("pip install errant -q")
os.system("python -m spacy download en -q")
print("✅ ERRANT installed!")


✅ ERRANT installed!


In [14]:
# ── Cell 13: ERRANT annotation + scoring ──────────────────────────────────────
import os, shutil

print("Running ERRANT annotation on dev predictions...")
ret = os.system(f"errant_parallel -orig {DEV_ORIG} -cor {DEV_PRED} -out {DEV_HYP_M2}")
print(f"errant_parallel exit code: {ret}")

if os.path.exists(DEV_HYP_M2):
    print("✅ dev_hyp.m2 created!")
else:
    print("❌ dev_hyp.m2 not created — check exit code above")
    for label, path in [("DEV_ORIG", DEV_ORIG), ("DEV_PRED", DEV_PRED)]:
        if os.path.exists(path):
            with open(path) as f:
                lines = sum(1 for _ in f)
            print(f"  {label}: {lines} lines → {path}")
        else:
            print(f"  ❌ {label} missing: {path}")

if not os.path.exists(DEV_HYP_M2):
    raise FileNotFoundError("dev_hyp.m2 not created — fix errant_parallel error above first")

with open(DEV_HYP_M2) as f:
    hyp_count = sum(1 for line in f if line.startswith("S "))
with open(GOLD_M2) as f:
    gold_count = sum(1 for line in f if line.startswith("S "))

print(f"hyp sentences:  {hyp_count}")
print(f"gold sentences: {gold_count}")

if hyp_count != gold_count:
    print("❌ Sentence count mismatch!")
else:
    print("\n========================================")
    print("ERRANT Scores on BEA-2019 Dev Set:")
    print("(Precision / Recall / F0.5)")
    print("========================================")
    result = os.popen(f"errant_compare -hyp {DEV_HYP_M2} -ref {GOLD_M2} 2>&1").read()
    print(result)

# ── Save to Drive ──
shutil.copy(DEV_HYP_M2, os.path.join(EVAL_OUT, "dev_hyp.m2"))
print(f"✅ dev_hyp.m2 saved to Drive: {EVAL_OUT}")


Running ERRANT annotation on dev predictions...
errant_parallel exit code: 0
✅ dev_hyp.m2 created!
hyp sentences:  4384
gold sentences: 4384

ERRANT Scores on BEA-2019 Dev Set:
(Precision / Recall / F0.5)

=========== Span-Based Correction ============
TP	FP	FN	Prec	Rec	F0.5
503	432	6958	0.538	0.0674	0.2245


✅ dev_hyp.m2 saved to Drive: /content/drive/Shareddrives/gec_thesis/gec_thesis/models/eval/tinybert


In [15]:
# ── Cell 14: Zip final model for download / archiving ─────────────────────────
import zipfile, os

zip_path = "/content/stage3_tinybert_final.zip"

files_to_zip = [
    (f"{STAGE3_DIR}/model_best.th",       "model_best.th"),
    (f"{WORK_DIR}/gector/gec_model.py",   "gec_model.py"),
    (f"{WORK_DIR}/predict.py",            "predict.py"),
]

with zipfile.ZipFile(zip_path, "w") as z:
    for src, arcname in files_to_zip:
        if os.path.exists(src):
            z.write(src, arcname)
            print(f"✅ Added {arcname}")
        else:
            print(f"❌ MISSING: {src}")

    # Add output_vocabulary folder
    for root, dirs, files in os.walk(VOCAB_PATH):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.join("output_vocabulary", os.path.relpath(full_path, VOCAB_PATH))
            z.write(full_path, arcname)
    print("✅ Added output_vocabulary/")

size = os.path.getsize(zip_path) / 1e6
print(f"\n✅ Done! stage3_tinybert_final.zip ({size:.1f} MB) → {zip_path}")

# Also copy to Drive
import shutil
drive_zip = os.path.join(EVAL_OUT, "stage3_tinybert_final.zip")
shutil.copy(zip_path, drive_zip)
print(f"✅ Also saved to Drive: {drive_zip}")


✅ Added model_best.th
✅ Added gec_model.py
✅ Added predict.py
✅ Added output_vocabulary/

✅ Done! stage3_tinybert_final.zip (63.8 MB) → /content/stage3_tinybert_final.zip
✅ Also saved to Drive: /content/drive/Shareddrives/gec_thesis/gec_thesis/models/eval/tinybert/stage3_tinybert_final.zip
